In [ ]:
import scanpy as sc
import numpy as np
# adata是退化的,cdata是插补后的直接结果
adata = sc.read_h5ad('/root/autodl-tmp/degraded_CESC_visium.h5ad')
cdata = sc.read_h5ad('/root/autodl-tmp/transfer/CESC_visium_hd_2um.h5ad')

In [2]:
import anndata as ad

def align_anndata_efficient(cdata, adata):
    """稀疏矩阵严格对齐方法"""
    
    common_obs = cdata.obs_names.intersection(adata.obs_names)
    common_vars = cdata.var_names.intersection(adata.var_names)
    
    def _subset_anndata(target_ad):
        # 1. 获取整数索引
        obs_idx = target_ad.obs_names.get_indexer(common_obs)
        var_idx = target_ad.var_names.get_indexer(common_vars)
        
        # 2. 分步切片稀疏矩阵 (先切行，再切列)
        X_sparse = target_ad.X.tocsr()
        X_sub = X_sparse[obs_idx, :]
        X_sub = X_sub[:, var_idx]
        
        # 3. 切片元数据与空间坐标
        obs_sub = target_ad.obs.iloc[obs_idx].copy()
        var_sub = target_ad.var.iloc[var_idx].copy()
        obsm_sub = {k: v[obs_idx] for k, v in target_ad.obsm.items()}
        
        # 4. 重新实例化
        return ad.AnnData(X=X_sub, obs=obs_sub, var=var_sub, obsm=obsm_sub)

    cdata_aligned = _subset_anndata(cdata)
    adata_aligned = _subset_anndata(adata)
    
    return cdata_aligned, adata_aligned

cdata, adata = align_anndata_efficient(cdata, adata)

In [3]:
import joblib
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# ==========================================
# 模块 1：基准拟合与降维 (纯稀疏矩阵运算)
# ==========================================
def fit_transform_clean(cdata_filtered, save_path="transforms.pkl"):
    """直接对稀疏矩阵进行 SVD 降维，随后在低维空间标准化"""
    
    # 1. 保持稀疏格式直接输入
    X_sparse = cdata_filtered.X
    
    # 2. 使用 TruncatedSVD 替代 PCA，输出 (N, 64) 稠密矩阵
    svd = TruncatedSVD(n_components=64)
    X_svd = svd.fit_transform(X_sparse)
    
    # 3. 对 64维 小矩阵进行标准化
    scaler = StandardScaler()
    X_svd_norm = scaler.fit_transform(X_svd)
    
    cdata_filtered.obsm['X_pca_norm'] = X_svd_norm
    joblib.dump({'pca': svd, 'scaler': scaler}, save_path)  # 键名保持 pca 以兼容后续代码
    
    return cdata_filtered

# ==========================================
# 模块 2：退化数据映射 (纯稀疏矩阵运算)
# ==========================================
def transform_degraded(adata_filtered, model_path="transforms.pkl"):
    """使用已保存的模型对稀疏的 Degraded Data 进行特征映射"""
    
    models = joblib.load(model_path)
    svd = models['pca']
    scaler = models['scaler']
    
    # 1. 保持稀疏格式
    X_sparse = adata_filtered.X
    
    # 2. 稀疏矩阵直接投影
    X_svd = svd.transform(X_sparse)
    
    # 3. 低维空间标准化
    X_svd_norm = scaler.transform(X_svd)
    
    adata_filtered.obsm['X_pca_norm'] = X_svd_norm
    
    return adata_filtered

In [ ]:
# ==========================================
# 阶段一：预处理 (模型推理前执行)
# ==========================================
# 假设 cdata 和 adata 已经加载到内存中
MODEL_SAVE_PATH = "CESC_transforms.pkl"

# 2. 拟合 Clean Data 并保存模型
cdata_filtered = fit_transform_clean(cdata, MODEL_SAVE_PATH)

# 3. 映射 Degraded Data
adata_filtered = transform_degraded(adata, MODEL_SAVE_PATH)

In [ ]:
cdata_filtered.write_h5ad('/root/autodl-tmp/SVD_CESC_visium_hd_2um.h5ad')
adata_filtered.write_h5ad('/root/autodl-tmp/SVD_degraded_CESC_visium_hd_2um.h5ad')

In [ ]:
import os
import glob
import torch
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix

# ==========================================
# 模块 3：张量解析与组装
# ==========================================
def assemble_tensor_to_anndata(pt_dir):
    """从 .pt 文件中提取有效特征并组装为 64维的 AnnData"""
    pt_files = glob.glob(os.path.join(pt_dir, "*.pt"))
    all_features = []
    
    for filepath in pt_files:
        data = torch.load(filepath)
        pred_expr = data['pred_expr']       # [64, 512, 512]
        pred_cell_id = data['pred_cell_id'] # [1, 512, 512]
        
        mask = pred_cell_id[0] > 0
        valid_features = pred_expr[:, mask].T.numpy()
        all_features.append(valid_features)
    
    X_total = np.vstack(all_features)
    ddata = ad.AnnData(X=X_total)
    
    return ddata

# ==========================================
# 模块 4：特征升维与还原
# ==========================================
def inverse_transform_denoised(ddata, model_path="transforms.pkl"):
    """将 64 维去噪数据还原至原始高维基因空间"""
    models = joblib.load(model_path)
    pca = models['pca']
    scaler = models['scaler']
    
    X_64d = ddata.X
    
    X_pca_inv = scaler.inverse_transform(X_64d)
    X_original_space = pca.inverse_transform(X_pca_inv)
    
    ddata.X = csr_matrix(X_original_space)
    
    return ddata

In [ ]:
# ==========================================
# 阶段二：后处理 (模型推理后执行)
# ==========================================
PT_OUTPUT_DIR = "/path/to/your/model_outputs"

# 4. 从模型输出的 .pt 文件组装 AnnData
ddata = assemble_tensor_to_anndata(PT_OUTPUT_DIR)

# 5. 特征还原回 1696 维稀疏矩阵
ddata = inverse_transform_denoised(ddata, MODEL_SAVE_PATH)

print(f"去噪还原后的数据维度: {ddata.shape}")
print(f"数据格式: {type(ddata.X)}")